[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-02-actors-object-store.ipynb#scrollTo=b1c2d3e4)

---
# Day 2 · Actors and the Object Store — Stateful Distributed Computing
**certified-journeys / ray-certified** · Day 2 · Core Patterns

> **Goal for today:** Build stateful distributed services using Ray Actors, efficiently share large data with `ray.put()` and the Plasma object store, and scale throughput with `ActorPool`.


In [ ]:
%pip install -q 'ray[default]' numpy


## Step 1 · What Is a Ray Actor?

A Ray **Actor** is a `@ray.remote`-decorated **class** — it runs as a long-lived worker process that holds **mutable state** between method calls. This contrasts with Ray tasks (stateless functions):

| | Ray Task (`@ray.remote` function) | Ray Actor (`@ray.remote` class) |
|---|---|---|
| State | Stateless | Stateful (instance variables) |
| Lifetime | Ends when function returns | Lives until `.kill()` or GC |
| Concurrency | Parallel by default | Single-threaded (queued) by default |
| Use case | Pure computation, map/reduce | Counters, caches, model servers, queues |

Actors are the foundation of Ray Serve (model serving) and Ray's parameter server pattern for distributed ML training.


In [ ]:
import ray
import time
import numpy as np

ray.init(ignore_reinit_error=True)

print(f"Ray {ray.__version__} ready — CPUs available: {ray.available_resources().get('CPU', 0):.0f}")


### What just happened?
- Ray is initialised with a local cluster — same as Day 1.
- We import `numpy` because Day 2 uses it to demonstrate the Plasma object store with large arrays.
- The local cluster will host both Actor processes and the shared-memory Plasma object store.


## Step 2 · Defining and Instantiating an Actor

An Actor is any Python class decorated with `@ray.remote`. Instantiate it with `.remote()` instead of the normal constructor — this creates the actor **in a dedicated worker process**.

```python
@ray.remote
class MyActor:
    def __init__(self, initial_value):
        self.value = initial_value

    def method(self, arg):
        self.value += arg
        return self.value

# Instantiate — returns an ActorHandle, not the object itself
actor = MyActor.remote(initial_value=0)

# Call methods — each returns an ObjectRef
ref = actor.method.remote(5)
result = ray.get(ref)   # → 5
```


In [ ]:
@ray.remote
class Counter:
    """A simple stateful counter actor."""

    def __init__(self, name: str, start: int = 0):
        self.name  = name
        self.count = start

    def increment(self, amount: int = 1) -> int:
        self.count += amount
        return self.count

    def decrement(self, amount: int = 1) -> int:
        self.count -= amount
        return self.count

    def get_count(self) -> int:
        return self.count

    def reset(self) -> None:
        self.count = 0

# Create two independent actor instances — each runs in its own process
page_views  = Counter.remote(name="page_views",  start=0)
error_count = Counter.remote(name="error_count", start=0)

# Call methods — they return ObjectRefs
ref1 = page_views.increment.remote(10)
ref2 = page_views.increment.remote(5)
ref3 = error_count.increment.remote(1)

# Collect results
r1, r2, r3 = ray.get([ref1, ref2, ref3])
print(f"page_views  after +10      : {r1}")
print(f"page_views  after +5       : {r2}")
print(f"error_count after +1       : {r3}")

# Current state
final_pv, final_ec = ray.get([page_views.get_count.remote(), error_count.get_count.remote()])
print(f"\nFinal page_views  : {final_pv}")
print(f"Final error_count : {final_ec}")


### What just happened?
- Each `Counter.remote()` call created a **separate process** with its own isolated state.
- Method calls are **queued** — `increment(10)` and `increment(5)` on the same actor run sequentially, so the final count is correct (no race condition).
- **ActorHandle** (`page_views`, `error_count`) is serialisable — you can pass it to tasks or other actors.
- The actor lives until the handle goes out of scope or you call `ray.kill(actor)`.


## Step 3 · 100 Concurrent Increments — Actor Method Queuing

Because actor methods are **queued** (single-threaded by default), you can safely fire 100 concurrent `increment()` calls and the final count will always equal 100. There is no lock needed — the actor is the synchronisation mechanism.

This is Ray's answer to the actor model (Erlang, Akka): **shared mutable state without locks**, because each actor is single-threaded internally while the caller is fully concurrent.


In [ ]:
@ray.remote
class SafeCounter:
    """Counter that is safe under concurrent access — no locks needed."""

    def __init__(self):
        self.count = 0
        self.history = []

    def increment(self) -> int:
        self.count += 1
        self.history.append(self.count)  # record each intermediate value
        return self.count

    def get_count(self) -> int:
        return self.count

    def get_history(self) -> list:
        return self.history

counter = SafeCounter.remote()

# Fire 100 concurrent increment calls — all submitted at once
N = 100
start = time.time()
refs = [counter.increment.remote() for _ in range(N)]

# Wait for all to complete
results = ray.get(refs)
elapsed = time.time() - start

final = ray.get(counter.get_count.remote())
print(f"Submitted    : {N} concurrent increments")
print(f"Final count  : {final}   (expected: {N})")
print(f"Elapsed      : {elapsed:.3f}s")
print(f"Last 10 return values: {results[-10:]}")

# Verify: history should contain every integer from 1 to N (no skips or repeats)
history = ray.get(counter.get_history.remote())
expected = list(range(1, N + 1))
print(f"History is 1..{N} in order: {sorted(history) == expected}")


### What just happened?
- **100 concurrent submits, zero race conditions** — the actor's internal queue serialised the calls.
- The `history` list has every integer from 1 to 100 in order — proving atomicity.
- The return values from `results` may be in any order because tasks complete in scheduling order, but the **state** is always consistent.
- Compare: a Python `threading.Thread`-based counter would need a `threading.Lock` to achieve the same correctness.


## Step 4 · The Plasma Object Store — `ray.put()` and Zero-Copy Sharing

Ray's **Plasma object store** is a shared-memory region that all worker processes on a node can access without copying. When you `ray.put(obj)`, the object is serialised (via Apache Arrow) and placed in Plasma. Any task that requests that `ObjectRef` reads directly from shared memory — **zero copies**.

| Scenario | Without `ray.put()` | With `ray.put()` |
|----------|--------------------|-----------------|
| 10 tasks, same 1 GB array | 10 GB transferred (one copy per task) | 1 GB in Plasma, 10 tasks read same memory |
| Serialisation | Per-call pickling | Once at `put()` time |
| Cross-node | Sends bytes over network | Plasma → object transfer (once per node) |


In [ ]:
@ray.remote
def process_array(arr_ref, row_slice: tuple) -> dict:
    """Process a slice of a large array — receives a shared ObjectRef."""
    arr = arr_ref   # Ray deserialises from Plasma (zero-copy on same node)
    start, end = row_slice
    chunk = arr[start:end]
    return {
        "rows": end - start,
        "mean": float(np.mean(chunk)),
        "std":  float(np.std(chunk)),
        "min":  float(np.min(chunk)),
        "max":  float(np.max(chunk)),
    }

# Create a large NumPy array (simulate a dataset too big to copy per-task)
big_array = np.random.rand(10_000, 100).astype(np.float32)   # ~4 MB
print(f"Array shape  : {big_array.shape}")
print(f"Array size   : {big_array.nbytes / 1e6:.1f} MB")

# Put the array in the object store ONCE
arr_ref = ray.put(big_array)
print(f"ObjectRef    : {arr_ref}")
print(f"Ref type     : {type(arr_ref).__name__}")

# Now fan out to 10 tasks — each reads from the SAME Plasma object
n_tasks = 10
rows_per_task = big_array.shape[0] // n_tasks
slices = [(i * rows_per_task, (i + 1) * rows_per_task) for i in range(n_tasks)]

refs = [process_array.remote(arr_ref, sl) for sl in slices]
stats = ray.get(refs)

print(f"\nPer-chunk statistics ({n_tasks} chunks):")
for i, s in enumerate(stats):
    print(f"  Chunk {i:02d}: rows={s['rows']}  mean={s['mean']:.4f}  std={s['std']:.4f}")

# Compare: passing the raw array (triggers a copy per task)
print("\nPassing raw array to tasks (not recommended for large arrays):")
refs_raw = [process_array.remote(big_array, sl) for sl in slices[:3]]
_ = ray.get(refs_raw)
print("  Done — each task received its own copy of the array.")
print("  Use ray.put() to avoid this for arrays > a few MB.")


### What just happened?
- `ray.put(big_array)` placed the array in Plasma once — all 10 tasks read the **same memory region**.
- **Passing the raw array** to tasks triggers a pickle + copy for each task — 10x more serialisation overhead.
- The `arr_ref` ObjectRef is tiny (a few bytes) — passing it to tasks is essentially free.
- On a real cluster, Ray transfers the Plasma object from the head node to each worker node **once per node**, not once per task.


## Step 5 · Named Actors — Accessing Actors Across Tasks

By default an actor lives only as long as its Python handle. With `name=` you register the actor in Ray's **Global Name Service (GNS)** — any task can look it up by name with `ray.get_actor(name)`.

Named actors are essential for singletons: a shared cache, a request counter, a model parameter store.


In [ ]:
@ray.remote
class SharedCache:
    """A simple key-value cache actor — accessible by name from anywhere in the cluster."""

    def __init__(self):
        self._store: dict = {}
        self._hits: int = 0
        self._misses: int = 0

    def put(self, key: str, value) -> None:
        self._store[key] = value

    def get(self, key: str):
        if key in self._store:
            self._hits += 1
            return self._store[key]
        self._misses += 1
        return None

    def stats(self) -> dict:
        return {
            "size":   len(self._store),
            "hits":   self._hits,
            "misses": self._misses,
            "hit_rate": round(self._hits / max(1, self._hits + self._misses), 3)
        }

# Create actor with a global name — lifetime='detached' survives driver restarts
cache = SharedCache.options(name="global_cache", lifetime="detached").remote()

@ray.remote
def worker_task(key: str, value: str = None) -> str:
    """Worker that reads/writes the named cache without an ActorHandle argument."""
    # Look up the actor by name inside a task — no handle passed as argument
    c = ray.get_actor("global_cache")
    if value is not None:
        ray.get(c.put.remote(key, value))
        return f"stored {key}={value}"
    else:
        result = ray.get(c.get.remote(key))
        return f"{key} -> {result}"

# Populate the cache from several tasks
write_refs = [
    worker_task.remote("model_version", "v2.1.0"),
    worker_task.remote("batch_size",    "256"),
    worker_task.remote("learning_rate", "0.001"),
]
print("Writes:", ray.get(write_refs))

# Read from the cache from several tasks
read_refs = [
    worker_task.remote("model_version"),
    worker_task.remote("batch_size"),
    worker_task.remote("missing_key"),   # cache miss
]
print("Reads:",  ray.get(read_refs))

print("Cache stats:", ray.get(cache.stats.remote()))

# Clean up detached actor manually
ray.kill(cache)


### What just happened?
- The `global_cache` actor was registered in Ray's GNS — tasks looked it up with `ray.get_actor()` without needing the handle passed as an argument.
- **`lifetime='detached'`** means the actor survives even if the Python driver exits — persistent across sessions.
- A cache miss returned `None` — the actor tracked hits and misses internally, thread-safe by design.
- **`ray.kill(actor)`** is required to terminate detached actors — they won't GC automatically.


## Step 6 · `ActorPool` — Throughput Scaling with a Pool of Workers

`ray.util.ActorPool` manages a **fixed pool of actor instances** and distributes work across them. It's ideal when:
- Each actor holds an expensive resource (a loaded ML model, a DB connection).
- You want to cap the number of live actors (bounded resource usage).
- You need a map-like API with actor-level state.

```python
from ray.util import ActorPool

pool = ActorPool([MyActor.remote() for _ in range(4)])

# map: apply fn(actor, value) to each item in the iterable
results = list(pool.map(lambda actor, x: actor.predict.remote(x), inputs))
```


In [ ]:
from ray.util import ActorPool

@ray.remote
class ModelWorker:
    """Simulates a worker that loads a model once and scores many batches."""

    def __init__(self, worker_id: int):
        self.worker_id = worker_id
        # Simulate expensive model loading (done ONCE per actor)
        time.sleep(0.2)
        self.model_loaded = True
        self.tasks_processed = 0

    def score(self, batch: list) -> dict:
        """Score a batch of inputs — fast because model is already loaded."""
        time.sleep(0.05)   # simulate fast inference
        self.tasks_processed += 1
        # Fake model output: multiply each input by 2.0
        scores = [round(x * 2.0, 3) for x in batch]
        return {
            "worker_id": self.worker_id,
            "batch_size": len(batch),
            "scores": scores,
            "tasks_done": self.tasks_processed,
        }

    def get_stats(self) -> dict:
        return {"worker_id": self.worker_id, "tasks_processed": self.tasks_processed}

# Create a pool of 3 workers — each loads the model once during __init__
pool_size = 3
pool = ActorPool([ModelWorker.remote(i) for i in range(pool_size)])
print(f"ActorPool created with {pool_size} workers")

# Generate 12 batches of input data
batches = [[float(i * 10 + j) for j in range(5)] for i in range(12)]

# pool.map distributes batches across the 3 workers automatically
results = list(pool.map(lambda actor, batch: actor.score.remote(batch), batches))

print(f"\nProcessed {len(results)} batches across {pool_size} workers:")
worker_loads = {}
for r in results:
    wid = r['worker_id']
    worker_loads[wid] = worker_loads.get(wid, 0) + 1

for wid, load in sorted(worker_loads.items()):
    print(f"  Worker {wid}: handled {load} batches")

print(f"\nSample result: {results[0]}")

# pool.map_unordered returns results as they complete (better throughput)
print("\nUsing map_unordered (results arrive as workers finish):")
count = 0
for result in pool.map_unordered(lambda a, b: a.score.remote(b), batches[:6]):
    count += 1
print(f"  Received {count} results via map_unordered")


### What just happened?
- `ActorPool` automatically **load-balanced** 12 batches across 3 workers — roughly 4 batches each.
- Each worker loaded the model **once** in `__init__` — far cheaper than loading per-task.
- `pool.map()` preserves **submission order** in results; `pool.map_unordered()` returns faster results first.
- This pattern directly mirrors Ray Serve's replica pool — understanding `ActorPool` means you understand Serve's internals.


## Step 7 · Actor Max Concurrency — Serving Actors

For serving/async use cases, you can allow multiple simultaneous method calls with `max_concurrency=N`. This requires either:
- **`asyncio`** inside the actor (non-blocking I/O), or
- Accepting that N method calls may run **interleaved** in the same thread.

Use `max_concurrency` for: HTTP request handlers, async DB writers, throughput-optimised caches.


In [ ]:
import asyncio

@ray.remote(max_concurrency=5)   # allow 5 simultaneous method calls
class AsyncRequestHandler:
    """Simulates an async HTTP handler — multiple requests handled concurrently."""

    def __init__(self):
        self.request_count = 0
        self.active = 0
        self.peak_active = 0

    async def handle_request(self, request_id: int, latency: float) -> dict:
        """Async method — uses await so other requests can proceed concurrently."""
        self.request_count += 1
        self.active += 1
        self.peak_active = max(self.peak_active, self.active)

        await asyncio.sleep(latency)   # non-blocking wait (vs time.sleep)

        self.active -= 1
        return {"request_id": request_id, "latency": latency, "status": 200}

    def get_stats(self) -> dict:
        return {
            "total_requests": self.request_count,
            "peak_concurrent": self.peak_active,
        }

handler = AsyncRequestHandler.remote()

# Fire 10 concurrent requests, each taking 0.3s
import random
random.seed(7)
latencies = [round(random.uniform(0.1, 0.4), 2) for _ in range(10)]

start = time.time()
refs = [handler.handle_request.remote(i, lat) for i, lat in enumerate(latencies)]
responses = ray.get(refs)
elapsed = time.time() - start

print(f"Handled {len(responses)} requests in {elapsed:.2f}s")
print(f"Sum of latencies: {sum(latencies):.2f}s  (would take this long without concurrency)")
stats = ray.get(handler.get_stats.remote())
print(f"Peak concurrent : {stats['peak_concurrent']}  (max_concurrency=5)")

# Compare: default single-threaded actor (max_concurrency=1)
@ray.remote(max_concurrency=1)
class SyncHandler:
    async def handle_request(self, request_id: int, latency: float) -> dict:
        await asyncio.sleep(latency)
        return {"request_id": request_id}

sync_handler = SyncHandler.remote()
start = time.time()
refs = [sync_handler.handle_request.remote(i, 0.1) for i in range(5)]
ray.get(refs)
sync_elapsed = time.time() - start
print(f"\nSync handler (max_concurrency=1): {sync_elapsed:.2f}s for 5 x 0.1s requests")
print(f"Async handler (max_concurrency=5): ~{5 * 0.1 / 5:.2f}s expected")


### What just happened?
- `max_concurrency=5` let 5 async method calls run simultaneously inside the same actor.
- Using `await asyncio.sleep()` (not `time.sleep()`) is what enables true concurrency — `time.sleep` would block the event loop.
- **Peak concurrent** shows how many requests overlapped — with `max_concurrency=1` it would always be 1.
- This is exactly how **Ray Serve** handles HTTP traffic — each replica is an actor with configurable `max_concurrency`.


In [ ]:
# Challenge: Build a distributed leaderboard
#
# Create an Actor called Leaderboard that:
#   1. Accepts score(name: str, score: float) method calls
#   2. Keeps only the HIGHEST score per player (update if new score is better)
#   3. Implements top_n(n: int) -> list[dict] that returns the top n players
#      as a sorted list of {"rank": i, "name": name, "score": score} dicts
#   4. Implements player_count() -> int
#
# Then:
#   - Create a pool of 20 score-submitting tasks (each submits a random score)
#   - Submit all tasks concurrently using .remote()
#   - Collect results with ray.get()
#   - Print the top 5 players
#
# Players: ["Alice", "Bob", "Carol", "Dan", "Eve"]
# Scores: random floats 0–100

# Your solution here:
# @ray.remote
# class Leaderboard:
#     def __init__(self): ...
#     def score(self, name, score): ...
#     def top_n(self, n): ...
#     def player_count(self): ...


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `@ray.remote` class | Defines an Actor — runs as a dedicated long-lived process |
| `Actor.remote()` | Creates the actor instance; returns an `ActorHandle` |
| Method queuing | Calls are serialised by default — no locks needed for state mutations |
| `ray.put(obj)` | Places object in Plasma store once; returns `ObjectRef` |
| Plasma store | Shared memory on each node — zero-copy reads for co-located tasks |
| Named actors | `Actor.options(name=...).remote()` + `ray.get_actor(name)` for global access |
| `ActorPool` | Fixed pool of actors; `map()` / `map_unordered()` for batch throughput |
| `max_concurrency` | Allow N simultaneous async method calls in one actor |

> **Tip:** Actors are single-threaded by default — each method call is queued. Use `@ray.remote(max_concurrency=N)` to allow N simultaneous method calls for serving actors.

---
## What's next
**Day 3** → Ray Clusters — local, Docker, and Kubernetes setup. Understanding head nodes, worker nodes, the GCS, and how to write cluster YAML configs.

Mark Day 2 complete in your [tracker](../index.html).


In [ ]:
# Shut down Ray cleanly at the end of the notebook
ray.shutdown()
print("Ray cluster shut down.")
